In [ ]:
from tqdm import tqdm
from glob import glob
import pandas as pd
import numpy as np
import json

models = glob('../openLLMLeaderboard/*.pkl')

RULES = { #defines input and metric for each task
    tuple(sorted(['level', 'type', 'problem', 'solution', 'answer'])):
        (['problem'], 'exact_match'),

    tuple(sorted(['input','target'])):
        (['input'], 'acc_norm'),

    tuple(sorted(['question', 'answer'])):
        (['question'], 'exact_match'),

    tuple(sorted(['Pre-Revision Question', 'Pre-Revision Correct Answer', 'Pre-Revision Incorrect Answer 1', 'Pre-Revision Incorrect Answer 2', 'Pre-Revision Incorrect Answer 3', 'Pre-Revision Explanation', 'Self-reported question-writing time (minutes)', 'Question', 'Correct Answer', 'Incorrect Answer 1', 'Incorrect Answer 2', 'Incorrect Answer 3', 'Explanation', 'Revision Comments (from Question Writer)', 'Subdomain', "Writer's Difficulty Estimate", 'Extra Revised Question', 'Extra Revised Explanation', 'Extra Revised Correct Answer', 'Extra Revised Incorrect Answer 1', 'Extra Revised Incorrect Answer 2', 'Extra Revised Incorrect Answer 3', 'Non-Expert Validator Accuracy', 'Majority Non-Expert Vals Incorrect', 'Expert Validator Accuracy', 'Record ID', 'High-level domain', 'Question Writer', 'Feedback_EV_1', 'Validator Revision Suggestion_EV_1', 'Is First Validation_EV_1', 'Post hoc agreement_EV_1', 'Sufficient Expertise?_EV_1', 'Understand the question?_EV_1', 'Question Difficulty_EV_1', 'Validator Answered Correctly_EV_1', 'Self-reported time (minutes)_EV_1', 'Probability Correct_EV_1', 'Manual Correctness Adjustment_EV_1', 'Expert Validator_EV_1', 'Feedback_EV_2', 'Validator Revision Suggestion_EV_2', 'Is First Validation_EV_2', 'Post hoc agreement_EV_2', 'Sufficient Expertise?_EV_2', 'Understand the question?_EV_2', 'Question Difficulty_EV_2', 'Validator Answered Correctly_EV_2', 'Self-reported time (minutes)_EV_2', 'Probability Correct_EV_2', 'Manual Correctness Adjustment_EV_2', 'Expert Validator_EV_2', 'Feedback_NEV_1', 'Validator Answered Correctly_NEV_1', 'Explanation_NEV_1', 'Self-reported time (minutes)_NEV_1', 'Websites visited_NEV_1', 'Probability Correct_NEV_1', 'Manual Correctness Adjustment_NEV_1', 'Non-Expert Validator_NEV_1', 'Feedback_NEV_2', 'Validator Answered Correctly_NEV_2', 'Explanation_NEV_2', 'Self-reported time (minutes)_NEV_2', 'Websites visited_NEV_2', 'Probability Correct_NEV_2', 'Manual Correctness Adjustment_NEV_2', 'Non-Expert Validator_NEV_2', 'Feedback_NEV_3', 'Validator Answered Correctly_NEV_3', 'Explanation_NEV_3', 'Self-reported time (minutes)_NEV_3', 'Websites visited_NEV_3', 'Probability Correct_NEV_3', 'Manual Correctness Adjustment_NEV_3', 'Non-Expert Validator_NEV_3', 'Expert Validator Disagreement Category', 'Canary String', 'choice1', 'choice2', 'choice3', 'choice4', 'answer'])):
        (['Question','choice1','choice2','choice3','choice4'], 'acc_norm'),

    tuple(sorted(['question_id', 'question', 'options', 'answer', 'answer_index', 'cot_content', 'category', 'src'])):
        (['question','options'], 'acc'),

    tuple(sorted(['key', 'prompt', 'instruction_id_list', 'kwargs'])):
        (['prompt'], 'prompt_level_loose_acc'),

    tuple(sorted(['narrative', 'question', 'choices', 'answer_index', 'answer_choice'])):
        (['narrative','question','choices'], 'acc_norm'),

    tuple(sorted(['section_id','passage','question','query_id','id','answers','answer.number','answer.date.day','answer.date.month','answer.date.year','answer.spans','answer.worker_id','answer.hit_id','validated_answers.number','validated_answers.date','validated_answers.spans','validated_answers.worker_id','validated_answers.hit_id'])):
        (['passage', 'question'], 'em'),

    tuple(sorted(['id', 'question', 'answerKey', 'choices.text', 'choices.label'])):
        (['question', 'choices.text'], 'acc'), 
}

def get_doc_keys(x):
    if not isinstance(x, dict):
        return []
    return list(pd.json_normalize(x).columns)
    
def _stringify_value(v):
    if isinstance(v, list):
        items = []
        for x in v:
            if isinstance(x, (dict, list)):
                items.append(json.dumps(x, ensure_ascii=False))
            else:
                items.append(str(x))
        return " | ".join(items)
    elif isinstance(v, dict):
        return json.dumps(v, ensure_ascii=False)
    else:
        return "" if v is None else str(v)

def to_binary(x):
    if isinstance(x, str):
        return 1 if x.strip().lower() == "true" else 0
    return 1 if float(x) else 0

def build_input_and_metrics(row):
    doc = row['doc']
    keys = row['doc_keys']
    try:
        rule = RULES[tuple(sorted(keys))]
    except:
        print(row)

    input_fields, metric_name = rule
    metric = row[metric_name]

    chunks = []
    for field in input_fields:
        if '.' in field:
            temp = field.split('.')
            val = doc.get(temp[0]).get(temp[1])
        else:
            val = doc.get(field, None)
        s = _stringify_value(val)
        if s != "":
            chunks.append(f"{field}: {s}")
        else:
            chunks.append(f"{field}: ")
    input_str = " ".join(chunks)

    return pd.Series({"input": input_str, "correctness": to_binary(metric)})

In [ ]:
for model in tqdm(models):
    df = pd.read_pickle(model)
    df['doc_keys'] = df['doc'].apply(get_doc_keys)
    df[['input', 'correctness']] = df.apply(build_input_and_metrics, axis=1)
    df.to_pickle(model)